# Árbol enriquecido — lectura funcional con pandas
Mismo algoritmo, otra organización. El notebook original queda intacto.
Leé primero **3. Entrenar** y **4. Crecer**; los auxiliares vienen después.
Ejecutá todo en orden. Instalación en el kernel: `%pip install -r requirements.txt`.

In [ ]:
from math import expm1, isfinite, isnan, log, log2, sqrt
import pandas as pd


## 1. Datos por nombre
Pandas se usa para preparar e inspeccionar tablas, no para la recursión.
Elegí la clase y los atributos continuos explícitamente; no se infieren por su tipo.

In [ ]:
datos = pd.DataFrame(
    [
        ['sunny', 'hot', 'high', 'weak', 'no'],
        ['sunny', 'hot', 'high', 'strong', 'no'],
        ['overcast', 'hot', 'high', 'weak', 'yes'],
        ['rain', 'mild', 'high', 'weak', 'yes'],
        ['rain', 'cool', 'normal', 'weak', 'yes'],
        ['rain', 'cool', 'normal', 'strong', 'no'],
        ['overcast', 'cool', 'normal', 'strong', 'yes'],
        ['sunny', 'mild', 'high', 'weak', 'no'],
        ['sunny', 'cool', 'normal', 'weak', 'yes'],
        ['rain', 'mild', 'normal', 'weak', 'yes'],
        ['sunny', 'mild', 'normal', 'strong', 'yes'],
        ['overcast', 'mild', 'high', 'strong', 'yes'],
        ['overcast', 'hot', 'normal', 'weak', 'yes'],
        ['rain', 'mild', 'high', 'strong', 'no'],
    ],
    columns=['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play']
)
objetivo = "Play"
atributos_elegidos = ["Outlook", "Temperature", "Humidity", "Wind"]
continuos_elegidos = []
datos.head()


In [ ]:
faltantes = datos.replace("?", pd.NA).isna().sum()
distribucion_clases = datos[objetivo].value_counts(sort=False)
pd.DataFrame({"faltantes": faltantes, "tipo": datos.dtypes.astype(str)})


In [ ]:
distribucion_clases


## 2. De tabla a filas
Seleccionamos columnas por nombre y dejamos la clase al final, sin modificar la tabla.
`None`, `NaN`, `pd.NA` y `"?"` se normalizan a `None`; la clase no puede faltar.

In [ ]:
def preparar_dataframe(datos, objetivo, atributos=None, continuos=()):
    """Convierte una tabla a la entrada del núcleo, conservando el orden elegido."""
    if not isinstance(datos, pd.DataFrame) or datos.empty:
        raise ValueError("Se requiere un DataFrame no vacío.")
    if not datos.columns.is_unique:
        raise ValueError("Las columnas deben ser únicas.")
    if objetivo not in datos.columns:
        raise ValueError("La columna objetivo no existe.")
    atributos = list(datos.columns.drop(objetivo)) if atributos is None else list(atributos)
    continuos = list(continuos)
    if (len(set(atributos)) != len(atributos) or objetivo in atributos
            or any(nombre not in datos.columns for nombre in atributos)):
        raise ValueError("Los atributos deben existir, ser únicos y excluir la clase.")
    if (len(set(continuos)) != len(continuos)
            or any(nombre not in atributos for nombre in continuos)):
        raise ValueError("Los continuos deben ser atributos seleccionados y únicos.")
    nombres = atributos + [objetivo]
    tabla = datos.loc[:, nombres].astype(object).replace("?", None)
    tabla = tabla.where(pd.notna(tabla), None)
    if tabla[objetivo].isna().any():
        raise ValueError("La clase debe ser conocida.")
    filas = tabla.to_numpy().tolist()
    indices = list(range(len(atributos)))
    indices_continuos = [atributos.index(nombre) for nombre in continuos]
    return filas, indices, indices_continuos, nombres


In [ ]:
D, A, continuos, columns = preparar_dataframe(
    datos, objetivo, atributos=atributos_elegidos, continuos=continuos_elegidos
)


## 3. Entrenar: validar → preparar → crecer → podar
La firma conserva las opciones originales. La poda se aplica después del crecimiento.
Los auxiliares se definen abajo; el ejemplo se ejecuta al final.

In [ ]:
def entrenar_arbol_enriquecido(filas, atributos, continuos=(), pesos=None,
                               min_gain=1e-12, min_peso_rama=2.0, confianza=0.25, podar=True,
                               max_depth=None, min_samples_split=2, *,
                               criterio='gain', filtro_ganancia_promedio=True,
                               min_gain_ratio=None, correccion_mdl=False,
                               min_samples_leaf=0, min_weight_leaf=0.0,
                               metodo_poda=None, subtree_raising=True, ccp_alpha=0.0,
                               filas_validacion=None, pesos_validacion=None):
    """Coordina las cuatro etapas sin modificar los datos del llamador."""
    # 1. Validar antes de construir el árbol.
    metodo_poda, pesos_validacion = _validar_entrenamiento(
        filas, criterio=criterio, filtro_ganancia_promedio=filtro_ganancia_promedio,
        correccion_mdl=correccion_mdl, min_gain_ratio=min_gain_ratio,
        min_samples_leaf=min_samples_leaf, min_weight_leaf=min_weight_leaf,
        podar=podar, metodo_poda=metodo_poda, subtree_raising=subtree_raising,
        ccp_alpha=ccp_alpha, filas_validacion=filas_validacion, pesos_validacion=pesos_validacion)

    # 2. Preparar una sola vez las referencias y el peso raíz.
    pesos, atributos, continuos, referencias = _preparar_entrenamiento(
        filas, atributos, continuos, pesos)

    # 3. Crecer con las mismas reglas de selección y parada.
    arbol = _crecer_enriquecido(
        filas, atributos, pesos, continuos, referencias, min_gain, min_peso_rama,
        max_depth=max_depth, min_samples_split=min_samples_split, profundidad=0, peso_raiz=sum(pesos),
        criterio=criterio, filtro_ganancia_promedio=filtro_ganancia_promedio,
        min_gain_ratio=min_gain_ratio, correccion_mdl=correccion_mdl,
        min_samples_leaf=min_samples_leaf, min_weight_leaf=min_weight_leaf)

    # 4. Podar y guardar el resumen antes/después.
    return _aplicar_poda(
        arbol, filas, pesos, metodo_poda, confianza, subtree_raising,
        ccp_alpha, filas_validacion, pesos_validacion)


## 4. Crecer: detener → elegir → dividir → repetir
Las salidas tempranas evitan anidar el algoritmo. Los continuos pueden reutilizarse.
`min_gain` usa ganancia cruda ponderada por peso nodo/raíz, no Gain Ratio ni MDL.

In [ ]:
def _crecer_enriquecido(filas, atributos, pesos, continuos, referencias, min_gain, minimo,
                        max_depth, min_samples_split, profundidad, peso_raiz, *,
                        criterio='gain', filtro_ganancia_promedio=True, min_gain_ratio=None,
                        correccion_mdl=False, min_samples_leaf=0, min_weight_leaf=0.0,
                        originales=None):
    """Construye un nodo y repite el mismo procedimiento en cada rama."""
    originales = [p > 0 for p in pesos] if originales is None else originales
    opciones = dict(criterio=criterio, filtro_ganancia_promedio=filtro_ganancia_promedio,
                    min_gain_ratio=min_gain_ratio, correccion_mdl=correccion_mdl,
                    min_samples_leaf=min_samples_leaf, min_weight_leaf=min_weight_leaf)
    nodo = NodoEnriquecido(filas, pesos)

    if nodo.total_weight == 0 or len(nodo.class_counts) == 1:
        return nodo
    if not atributos:
        return nodo
    if max_depth is not None and profundidad >= max_depth:
        return nodo
    if len(filas) < min_samples_split:
        return nodo
    if info(filas, pesos) == 0:
        return nodo

    mejor = _seleccionar_prueba_enriquecida(
        filas, atributos, pesos, continuos, referencias, minimo,
        originales=originales, **opciones)
    if mejor is None:
        return nodo

    atributo, prueba = mejor
    ganancia_global = sum(pesos) / peso_raiz * prueba['gain']
    if not (prueba['gain'] > 0 and ganancia_global >= min_gain):
        return nodo

    nodo.attribute, nodo.threshold = atributo, prueba['threshold']
    grupos, nodo.branch_probabilities = particionar_con_desconocidos(
        filas, atributo, pesos, nodo.threshold)
    restantes = [a for a in atributos if a != atributo or a in continuos]
    for rama, (grupo, pesos_grupo) in grupos.items():
        originales_grupo = _originales_rama_enriquecida(
            filas, pesos, originales, atributo, nodo.threshold, rama)
        nodo.children[rama] = _crecer_enriquecido(
            grupo, restantes, pesos_grupo, continuos, referencias, min_gain, minimo,
            max_depth=max_depth, min_samples_split=min_samples_split,
            profundidad=profundidad + 1, peso_raiz=peso_raiz,
            originales=originales_grupo, **opciones)
    return nodo


## 5. Pesos y desconocidos
Los desconocidos se reparten entre ramas; no forman una categoría nueva.
Las filas, pesos y banderas de original conservan su alineación.

In [ ]:
def es_desconocido(value):
    """Reconoce None, "?" y NaN como valores desconocidos."""
    return (value is None or (isinstance(value, str) and value == "?")
            or (isinstance(value, float) and isnan(value)))


def pesos_casos(rows, weights=None):
    """Prepara pesos separados, finitos y no negativos; por defecto usa uno por caso."""
    result = [1.0] * len(rows) if weights is None else [float(w) for w in weights]
    if len(result) != len(rows) or any(not isfinite(w) or w < 0 for w in result):
        raise ValueError("Provide one finite nonnegative weight per row.")
    return result


def frecuencias_clase(rows, weights=None):
    """Suma los pesos por clase sin modificar las filas ni sus pesos."""
    counts = {}
    for row, weight in zip(rows, pesos_casos(rows, weights)):
        if weight > 0:
            counts[row[-1]] = counts.get(row[-1], 0.0) + weight
    return counts


def probabilidades_clase(rows, weights=None):
    """Normaliza las frecuencias ponderadas de las clases."""
    counts = frecuencias_clase(rows, weights)
    total = sum(counts.values())
    return {label: count / total for label, count in counts.items()}


def info(rows, weights=None):
    """Calcula info(T), la entropía de clases, admitiendo pesos por caso."""
    return -sum(p * log2(p) for p in probabilidades_clase(rows, weights).values())


In [ ]:
def separar_casos_conocidos(rows, attribute, weights=None, threshold=None):
    """Separa las ramas conocidas de los casos con resultado desconocido."""
    groups, unknown = {}, []
    for row, weight in zip(rows, pesos_casos(rows, weights)):
        if weight == 0:
            continue
        value = row[attribute]
        if es_desconocido(value):
            unknown.append((row, weight))
        else:
            branch = value if threshold is None else value <= threshold
            group, group_weights = groups.setdefault(branch, ([], []))
            group.append(row)
            group_weights.append(weight)
    return groups, unknown


def particionar_con_desconocidos(rows, attribute, weights=None, threshold=None):
    """Reparte cada caso desconocido con peso w por la probabilidad de cada rama."""
    groups, unknown = separar_casos_conocidos(rows, attribute, weights, threshold)
    known_weight = sum(sum(w) for _, w in groups.values())
    if known_weight == 0:
        return {}, {}
    probabilities = {branch: sum(w) / known_weight for branch, (_, w) in groups.items()}
    for branch, (group, group_weights) in groups.items():
        for row, weight in unknown:
            group.append(row)
            group_weights.append(weight * probabilities[branch])
    return groups, probabilities


def _originales_rama_enriquecida(filas, pesos, originales, atributo, umbral, rama):
    """Alinea elegibilidad con particionar_con_desconocidos: conocidos y luego nulos."""
    conocidos = [original for fila, peso, original in zip(filas, pesos, originales)
                 if peso > 0 and not es_desconocido(fila[atributo])
                 and (fila[atributo] if umbral is None else fila[atributo] <= umbral) == rama]
    repartidos = [False for fila, peso in zip(filas, pesos)
                  if peso > 0 and es_desconocido(fila[atributo])]
    return conocidos + repartidos


## 6. Evaluar cortes
Se distinguen ganancia cruda, ajuste MDL y cociente de ganancia.
Los mínimos de hoja se comprueban antes de seleccionar candidatos.

In [ ]:
def evaluar_prueba(rows, attribute, weights=None, threshold=None):
    """Calcula gain(X), split info(X) y su cociente para una prueba."""
    groups, unknown = separar_casos_conocidos(rows, attribute, weights, threshold)
    masses = {branch: sum(w) for branch, (_, w) in groups.items()}
    known_weight = sum(masses.values())
    total = known_weight + sum(w for _, w in unknown)
    if len(groups) < 2 or known_weight == 0:
        return None
    known_rows = [row for group, _ in groups.values() for row in group]
    known_weights = [w for _, group_weights in groups.values() for w in group_weights]
    after = sum(masses[branch] / known_weight * info(group, group_weights)
                for branch, (group, group_weights) in groups.items())
    gain = known_weight / total * (info(known_rows, known_weights) - after)
    proportions = [mass / total for mass in masses.values()]
    if total > known_weight:
        proportions.append((total - known_weight) / total)
    split_info = -sum(p * log2(p) for p in proportions)
    return {"threshold": threshold, "gain": gain, "split_info": split_info,
            "gain_ratio": gain / split_info if split_info > 0 else 0.0, "branch_weights": masses,
            "branch_probabilities": {branch: mass / known_weight
                                     for branch, mass in masses.items()}}


def _validar_minimos_hoja(min_samples_leaf, min_weight_leaf):
    """Valida restricciones propias de factibilidad, no reglas de poda."""
    if (isinstance(min_samples_leaf, bool) or not isinstance(min_samples_leaf, int)
            or min_samples_leaf < 0):
        raise ValueError('min_samples_leaf debe ser un entero no negativo.')
    if not isfinite(min_weight_leaf) or min_weight_leaf < 0:
        raise ValueError('min_weight_leaf debe ser finito y no negativo.')


def _cumple_minimos_hoja(filas, atributo, pesos, prueba, originales,
                         min_samples_leaf, min_weight_leaf):
    """Cuenta originales conocidos; exige masa total tras repartir desconocidos."""
    conteos = dict.fromkeys(prueba['branch_weights'], 0)
    for fila, peso, original in zip(filas, pesos, originales):
        valor = fila[atributo]
        if original and peso > 0 and not es_desconocido(valor):
            rama = valor if prueba['threshold'] is None else valor <= prueba['threshold']
            conteos[rama] += 1
    grupos, _ = particionar_con_desconocidos(filas, atributo, pesos, prueba['threshold'])
    return all(conteos[rama] >= min_samples_leaf and sum(ponderaciones) >= min_weight_leaf
               for rama, (_, ponderaciones) in grupos.items())


In [ ]:
def evaluar_umbrales_continuos(rows, attribute, weights=None, reference_values=None,
                      min_branch_weight=2.0, *, correccion_mdl=False,
                      min_samples_leaf=0, min_weight_leaf=0.0, originales=None):
    """Devuelve las estadísticas de cortes binarios válidos, ordenados por umbral."""
    _validar_minimos_hoja(min_samples_leaf, min_weight_leaf)
    if not isinstance(correccion_mdl, bool):
        raise ValueError('correccion_mdl debe ser booleana.')
    if not isfinite(min_branch_weight) or min_branch_weight <= 0:
        raise ValueError("min_branch_weight must be finite and positive.")
    weights = pesos_casos(rows, weights)
    originales = [w > 0 for w in weights] if originales is None else originales
    values = {row[attribute] for row, w in zip(rows, weights)
              if w > 0 and not es_desconocido(row[attribute])}
    reference = values if reference_values is None else {
        value for value in reference_values if not es_desconocido(value)}
    if any(not isinstance(v, (int, float)) or not isfinite(v) for v in values | reference):
        raise ValueError("Continuous values must be finite numbers.")
    if not values <= reference:
        raise ValueError("reference_values must include every known positive-weight value.")
    ordered = sorted(values)
    candidates = []
    for left, right in zip(ordered, ordered[1:]):
        midpoint = left / 2 + right / 2
        threshold = max((v for v in reference if v < right and v <= midpoint), default=left)
        stats = evaluar_prueba(rows, attribute, weights, threshold)
        if (stats is not None and min(stats['branch_weights'].values()) >= min_branch_weight
                and _cumple_minimos_hoja(rows, attribute, weights, stats, originales,
                                         min_samples_leaf, min_weight_leaf)):
            candidates.append(stats)
    if not candidates:
        return []
    penalty = log2(len(candidates)) / sum(weights) if correccion_mdl else 0.0
    for stats in candidates:
        stats['mdl_penalty'] = penalty
        stats['gain_ajustada'] = stats['gain'] - penalty
        stats['gain_ratio'] = (stats['gain_ajustada'] / stats['split_info']
                               if stats['split_info'] > 0 else 0.0)
    return candidates


## 7. Elegir una prueba
Primero se elige un umbral por atributo; luego se comparan los atributos.
Los empates mantienen el primer candidato. El filtro promedio sólo afecta a Gain Ratio.

In [ ]:
def _seleccionar_prueba_enriquecida(filas, atributos, pesos, continuos, referencias, minimo, *,
                                    criterio='gain', filtro_ganancia_promedio=True,
                                    min_gain_ratio=None, correccion_mdl=False,
                                    min_samples_leaf=0, min_weight_leaf=0.0, originales=None):
    """Compara un representante factible por atributo; conserva el primer empate."""
    originales = [p > 0 for p in pesos] if originales is None else originales
    elegibles = []
    for atributo in atributos:
        if atributo in continuos:
            candidatos = evaluar_umbrales_continuos(
                filas, atributo, pesos, referencias[atributo], minimo,
                correccion_mdl=correccion_mdl, min_samples_leaf=min_samples_leaf,
                min_weight_leaf=min_weight_leaf, originales=originales)
        else:
            prueba = evaluar_prueba(filas, atributo, pesos)
            candidatos = []
            if prueba is not None and _cumple_minimos_hoja(
                    filas, atributo, pesos, prueba, originales, min_samples_leaf, min_weight_leaf):
                prueba.update(gain_ajustada=prueba['gain'], mdl_penalty=0.0)
                candidatos.append(prueba)
        if candidatos:
            prueba = max(candidatos, key=lambda c: c['gain'])
            elegibles.append((atributo, prueba))  # El promedio incluye ganancia cero.
    if not elegibles:
        return None
    if criterio == 'gain_ratio' and filtro_ganancia_promedio:
        promedio = sum(p['gain_ajustada'] for _, p in elegibles) / len(elegibles)
        elegibles = [(a, p) for a, p in elegibles if p['gain_ajustada'] >= promedio]
    elegibles = [(a, p) for a, p in elegibles if p['gain'] > 0
                 and (not correccion_mdl or p['gain_ajustada'] > 0)]
    if not elegibles:
        return None
    campo = 'gain_ratio' if criterio == 'gain_ratio' else 'gain_ajustada'
    mejor = max(elegibles, key=lambda candidato: candidato[1][campo])
    if min_gain_ratio is not None and mejor[1]['gain_ratio'] < min_gain_ratio:
        return None
    return mejor


## 8. Nodo y predicción
Los nulos combinan distribuciones; una categoría nueva usa el respaldo del nodo.
La clase mayoritaria y los empates conservan el comportamiento original.

In [ ]:
class NodoEnriquecido:
    """Estado ponderado propio; NUL, libro pp. 29–32 / PDF pp. 3–6."""
    def __init__(self, filas, pesos):
        self.class_counts = frecuencias_clase(filas, pesos)
        self.total_weight = sum(self.class_counts.values())
        self.class_probabilities = probabilidades_clase(filas, pesos)
        self.prediction = max(self.class_counts, key=self.class_counts.get, default=None)
        self.attribute = None
        self.threshold = None
        self.children = {}
        self.branch_probabilities = {}


def estadisticas_arbol_enriquecido(arbol):
    """Describe estructura sin entrenar: profundidad desde raíz cero, nodos y hojas."""
    if arbol.attribute is None:
        return {'profundidad': 0, 'nodos': 1, 'hojas': 1}
    hijos = [estadisticas_arbol_enriquecido(hijo) for hijo in arbol.children.values()]
    return {'profundidad': 1 + max(h['profundidad'] for h in hijos),
            'nodos': 1 + sum(h['nodos'] for h in hijos),
            'hojas': sum(h['hojas'] for h in hijos)}


In [ ]:
def combinar_probabilidades_clase(distributions, branch_probabilities):
    """Combina distribuciones de clases con las probabilidades de sus ramas."""
    combined = {}
    for branch, branch_probability in branch_probabilities.items():
        for label, probability in distributions[branch].items():
            combined[label] = combined.get(label, 0.0) + branch_probability * probability
    return combined


def predecir_probabilidades_enriquecido(arbol, fila):
    """Combina ramas ante nulos; NUL, libro pp. 30–33 / PDF pp. 4–7."""
    if arbol.attribute is None:
        return dict(arbol.class_probabilities)
    valor = fila[arbol.attribute]
    if es_desconocido(valor):
        distribuciones = {rama: predecir_probabilidades_enriquecido(hijo, fila)
                          for rama, hijo in arbol.children.items()}
        combinada = combinar_probabilidades_clase(distribuciones, arbol.branch_probabilities)
        total = sum(combinada.values())
        return {clase: p / total for clase, p in combinada.items()} if total else {}
    rama = valor if arbol.threshold is None else valor <= arbol.threshold
    if rama not in arbol.children:
        return dict(arbol.class_probabilities)
    return predecir_probabilidades_enriquecido(arbol.children[rama], fila)


def predecir_enriquecido(arbol, fila):
    """Elige la clase más probable; empate por orden del nodo, convención propia."""
    probabilidades = predecir_probabilidades_enriquecido(arbol, fila)
    return max(arbol.class_counts, key=lambda clase: probabilidades.get(clase, 0), default=None)


## 9. Poda pesimista
Compara conservar, reemplazar por hoja o elevar una rama; recalibra sobre la misma población.
El estimador es la aproximación original, no una inversión binomial exacta.

In [ ]:
def errores_estimados(total_weight, observed_errors, confidence=0.25):
    """Aproxima el conteo de errores N por U_CF(E,N), no la tasa U_CF sola."""
    n, e = total_weight, observed_errors
    if (not all(isfinite(v) for v in (n, e, confidence))
            or not 0 <= e <= n or not 0 < confidence < 1):
        raise ValueError("Require 0 <= errors <= weight and 0 < confidence < 1.")
    if n == 0:
        return 0.0
    zero_errors = -n * expm1(log(confidence) / n)
    if e == 0:
        return zero_errors
    if e == n:
        return n
    if e < 1:
        anchor = min(1.0, n)
        upper = errores_estimados(n, anchor, confidence)
        return zero_errors + e / anchor * (upper - zero_errors)
    if e + 0.5 >= n:
        return e + 0.67 * (n - e)
    levels = (0.0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.40, 1.0)
    deviations = (4.0, 3.09, 2.58, 2.33, 1.65, 1.28, 0.84, 0.25, 0.0)
    index = next(i for i in range(1, len(levels)) if confidence <= levels[i])
    fraction = (confidence - levels[index - 1]) / (levels[index] - levels[index - 1])
    z = deviations[index - 1] + fraction * (deviations[index] - deviations[index - 1])
    f = (e + 0.5) / n
    upper = (f + z * z / (2 * n) + z * sqrt(f * (1 - f) / n + z * z / (4 * n * n)))
    return n * upper / (1 + z * z / n)


def U_CF(E, N, CF=0.25):
    """Devuelve una aproximación a la tasa U_CF(E,N), no al número de errores."""
    estimated = errores_estimados(N, E, CF)
    return estimated / N if N else 0.0


def estadisticas_hoja(rows, weights=None, confidence=0.25):
    """Describe una hoja mayoritaria hipotética, sin crear ni modificar nodos."""
    counts = frecuencias_clase(rows, weights)
    total = sum(counts.values())
    prediction = max(counts, key=counts.get, default=None)
    errors = total - max(counts.values(), default=0.0)
    return {"prediction": prediction, "class_counts": counts, "total_weight": total,
            "observed_errors": errors,
            "estimated_errors": errores_estimados(total, errors, confidence)}


def elegir_poda(subtree_errors, leaf_errors, raised_branch_errors=None):
    """Compara errores del subárbol, la hoja y una rama elevada sobre los mismos casos."""
    candidates = [("keep", subtree_errors), ("leaf", leaf_errors)]
    if raised_branch_errors is not None:
        candidates.append(("raise", raised_branch_errors))
    return min(candidates, key=lambda candidate: candidate[1])[0]


In [ ]:
def _poblaciones_enriquecidas(nodo, filas, pesos):
    """Redirige la población completa; NUL pp. 29–32 / PDF 3–6; POD pp. 37–39 / PDF 1–3."""
    utilizables, ponderaciones, respaldo, pesos_respaldo = [], [], [], []
    for fila, peso in zip(filas, pesos):
        valor = fila[nodo.attribute]
        desconocido = es_desconocido(valor)
        rama = None if desconocido else (valor if nodo.threshold is None else valor <= nodo.threshold)
        if desconocido or rama in nodo.children:
            utilizables.append(fila)
            ponderaciones.append(peso)
        else:
            respaldo.append(fila)
            pesos_respaldo.append(peso)
    grupos, probabilidades = particionar_con_desconocidos(
        utilizables, nodo.attribute, ponderaciones, nodo.threshold)
    return grupos, probabilidades, respaldo, pesos_respaldo


def _error_respaldo_enriquecido(nodo, filas, pesos, confianza):
    """Estima el respaldo propio con la misma aproximación; POD pp. 40–42 / PDF 4–6."""
    errores = sum(p for f, p in zip(filas, pesos) if f[-1] != nodo.prediction)
    return errores_estimados(sum(pesos), errores, confianza)


def _recalibrar_enriquecido(plantilla, filas, pesos, confianza):
    """Copia y reestima sobre TODOS los casos; POD, libro pp. 37–43 / PDF pp. 1–7."""
    nodo = NodoEnriquecido(filas, pesos)
    hoja = estadisticas_hoja(filas, pesos, confianza)
    if plantilla.attribute is None or nodo.total_weight == 0:
        return nodo, hoja['estimated_errors']
    grupos, probabilidades, respaldo, pesos_respaldo = _poblaciones_enriquecidas(plantilla, filas, pesos)
    if not grupos:
        return nodo, hoja['estimated_errors']
    nodo.attribute, nodo.threshold = plantilla.attribute, plantilla.threshold
    nodo.branch_probabilities = probabilidades
    error = _error_respaldo_enriquecido(nodo, respaldo, pesos_respaldo, confianza)
    for rama, (grupo, ponderaciones) in grupos.items():
        hijo, estimacion = _recalibrar_enriquecido(
            plantilla.children[rama], grupo, ponderaciones, confianza)
        nodo.children[rama] = hijo
        error += estimacion
    return nodo, error


In [ ]:
def _podar_enriquecido(plantilla, filas, pesos, confianza, subtree_raising=True):
    """Poda ascendente real: hoja, conservación o elevación; POD pp. 37–43 / PDF 1–7."""
    nodo, error = _recalibrar_enriquecido(plantilla, filas, pesos, confianza)
    if nodo.attribute is None:
        return nodo, error
    grupos, _, respaldo, pesos_respaldo = _poblaciones_enriquecidas(nodo, filas, pesos)
    error = _error_respaldo_enriquecido(nodo, respaldo, pesos_respaldo, confianza)
    for rama, (grupo, ponderaciones) in grupos.items():
        nodo.children[rama], estimacion = _podar_enriquecido(
            nodo.children[rama], grupo, ponderaciones, confianza, subtree_raising)
        error += estimacion
    hoja = estadisticas_hoja(filas, pesos, confianza)
    elevada, error_elevada = None, None
    if subtree_raising:
        mas_usada = max(nodo.children.values(), key=lambda hijo: hijo.total_weight)
        elevada, error_elevada = _recalibrar_enriquecido(mas_usada, filas, pesos, confianza)
    decision = elegir_poda(error, hoja['estimated_errors'], error_elevada)
    if decision == 'leaf':
        return NodoEnriquecido(filas, pesos), hoja['estimated_errors']
    if decision == 'raise':
        return elevada, error_elevada
    return nodo, error


## 10. Poda por riesgo
Reduced-error usa validación independiente; cost-complexity usa error ponderado + alpha × hojas.
Ambas comparan el árbol completo en una pasada greedy; no es weakest-link CART.

In [ ]:
def _riesgo_enriquecido(arbol, filas, pesos):
    """Conteo ponderado 0-1 con el predictor público, no suma de errores de hojas."""
    return sum(peso for fila, peso in zip(filas, pesos)
               if predecir_enriquecido(arbol, fila) != fila[-1])


def _podar_por_riesgo_enriquecido(plantilla, filas, pesos, alpha=0.0):
    """Una pasada greedy ascendente; compara cada cambio sobre el árbol COMPLETO."""
    from copy import deepcopy
    arbol = deepcopy(plantilla)

    def costo():
        return (_riesgo_enriquecido(arbol, filas, pesos)
                + alpha * estadisticas_arbol_enriquecido(arbol)['hojas'])

    def recorrer(nodo):
        if nodo.attribute is None:
            return
        for hijo in nodo.children.values():
            recorrer(hijo)
        anterior = costo()
        estado = nodo.attribute, nodo.threshold, nodo.children, nodo.branch_probabilities
        nodo.attribute, nodo.threshold, nodo.children, nodo.branch_probabilities = None, None, {}, {}
        if costo() > anterior:
            nodo.attribute, nodo.threshold, nodo.children, nodo.branch_probabilities = estado

    recorrer(arbol)
    return arbol


## 11. Validación y preparación internas
Se conservan las validaciones originales. Reduced-error exige datos independientes explícitos.
Nunca uses el conjunto de prueba final para podar.

In [ ]:
def _validar_poda_enriquecida(filas, podar, metodo, subtree_raising, alpha, validacion, pesos):
    """Resuelve compatibilidad y exige validación explícita antes de crecer."""
    if not isinstance(podar, bool) or not isinstance(subtree_raising, bool):
        raise ValueError('podar y subtree_raising deben ser booleanos.')
    if metodo is None:
        metodo = 'pesimista' if podar else 'ninguna'
    if metodo not in ('pesimista', 'reduced_error', 'cost_complexity', 'ninguna'):
        raise ValueError('metodo_poda debe ser pesimista, reduced_error, cost_complexity o ninguna.')
    if not podar and metodo != 'ninguna':
        raise ValueError('podar=False sólo admite metodo_poda=None o ninguna.')
    if (isinstance(alpha, bool) or not isinstance(alpha, (int, float))
            or not isfinite(alpha) or alpha < 0):
        raise ValueError('ccp_alpha debe ser finito y no negativo.')
    if metodo != 'reduced_error':
        if validacion is not None or pesos is not None:
            raise ValueError('filas_validacion y pesos_validacion sólo se admiten con reduced_error.')
        return metodo, None
    if validacion is filas:
        raise ValueError('La validación debe ser independiente del entrenamiento.')
    if validacion is None or len(validacion) == 0:
        raise ValueError('reduced_error requiere filas_validacion explícitas y no vacías.')
    if (not filas or any(len(fila) != len(filas[0]) for fila in validacion)
            or len(filas[0]) == 0):
        raise ValueError('La validación debe tener el mismo ancho que el entrenamiento, con clase al final.')
    ponderaciones = pesos_casos(validacion, pesos)
    if not isfinite(sum(ponderaciones)) or sum(ponderaciones) <= 0:
        raise ValueError('La validación debe tener peso total positivo y finito.')
    return metodo, ponderaciones


In [ ]:
def _validar_entrenamiento(filas, *, criterio, filtro_ganancia_promedio, correccion_mdl,
                           min_gain_ratio, min_samples_leaf, min_weight_leaf, podar,
                           metodo_poda, subtree_raising, ccp_alpha, filas_validacion,
                           pesos_validacion):
    """Rechaza opciones incompatibles antes de iniciar el crecimiento."""
    _validar_minimos_hoja(min_samples_leaf, min_weight_leaf)
    if criterio not in ('gain', 'gain_ratio'):
        raise ValueError('criterio debe ser gain o gain_ratio.')
    if not isinstance(filtro_ganancia_promedio, bool) or not isinstance(correccion_mdl, bool):
        raise ValueError('filtro_ganancia_promedio y correccion_mdl deben ser booleanos.')
    if min_gain_ratio is not None and (not isfinite(min_gain_ratio) or min_gain_ratio < 0):
        raise ValueError('min_gain_ratio debe ser None o finito no negativo.')
    return _validar_poda_enriquecida(
        filas, podar, metodo_poda, subtree_raising, ccp_alpha, filas_validacion, pesos_validacion)


def _preparar_entrenamiento(filas, atributos, continuos, pesos):
    """Prepara copias de opciones y referencias globales para los cortes continuos."""
    pesos = pesos_casos(filas, pesos)
    atributos, continuos = list(atributos), set(continuos)
    referencias = {a: [fila[a] for fila in filas] for a in continuos}
    return pesos, atributos, continuos, referencias


def _aplicar_poda(arbol, filas, pesos, metodo, confianza, subtree_raising,
                  ccp_alpha, filas_validacion, pesos_validacion):
    """Aplica el método elegido y adjunta estadísticas sin reentrenar."""
    antes = estadisticas_arbol_enriquecido(arbol)
    if metodo == 'pesimista':
        arbol = _podar_enriquecido(arbol, filas, pesos, confianza, subtree_raising)[0]
    elif metodo == 'reduced_error':
        arbol = _podar_por_riesgo_enriquecido(arbol, filas_validacion, pesos_validacion)
    elif metodo == 'cost_complexity':
        arbol = _podar_por_riesgo_enriquecido(arbol, filas, pesos, ccp_alpha)
    arbol.resumen_poda = {'antes': antes, 'despues': estadisticas_arbol_enriquecido(arbol),
                        'metodo': metodo}
    return arbol


## 12. Probar y comparar
Modificá las opciones de esta llamada y compará con `decision-tree-training.ipynb`.
Los resultados muestran estructura y predicciones, no exactitud de prueba.

In [ ]:
arbol_enriquecido = entrenar_arbol_enriquecido(D, A, continuos=continuos)
resumen_poda = pd.DataFrame({
    "antes": arbol_enriquecido.resumen_poda["antes"],
    "despues": arbol_enriquecido.resumen_poda["despues"],
}).T
resumen_poda


In [ ]:
predicciones = pd.DataFrame({
    "clase_real": datos[objetivo].tolist(),
    "prediccion": [predecir_enriquecido(arbol_enriquecido, fila) for fila in D],
})
predicciones


## Referencias
C4.5: continuos (pp. 25–26), desconocidos (pp. 27–33), poda (pp. 37–43).
MDL: [Quinlan, 1996](https://www.cs.cmu.edu/afs/cs/project/jair/pub/volume4/quinlan96a.pdf), con adaptación ponderada.
Contratos y fuentes detalladas: [notebook original](decision-tree-training.ipynb).